# 01. Анализ датасета фильмов

Цель ноутбука — показать структуру основного очищенного датасета `data/processed/output.csv`, оценить качество данных и понять, насколько он подходит для задачи прогнозирования кассовой выручки фильма.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_PATH = PROJECT_ROOT / "data" / "output.csv"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_csv(DATASET_PATH)
df.head()

## Общая структура

Основной файл должен содержать признаки фильма и целевую переменную `gross`, которая отражает кассовую выручку.

In [ ]:
print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} колонок")
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(), "missing_pct": df.isna().mean() * 100}))

print("Полные дубликаты:", df.duplicated().sum())
print("Дубликаты по name + year:", df.duplicated(["name", "year"]).sum())

## Числовые признаки

Проверим диапазоны года, оценки, голосов, бюджета, кассы и длительности фильма.

In [ ]:
numeric_columns = ["year", "score", "votes", "budget", "gross", "runtime"]
df[numeric_columns].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]).T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(np.log1p(df["budget"]), bins=40, color="#ff8a5b", edgecolor="white")
axes[0].set_title("Распределение log1p(budget)")
axes[0].set_xlabel("log1p(budget)")

axes[1].hist(np.log1p(df["gross"]), bins=40, color="#2ec4b6", edgecolor="white")
axes[1].set_title("Распределение log1p(gross)")
axes[1].set_xlabel("log1p(gross)")

plt.tight_layout()
plt.show()

## Категориальные признаки

Посмотрим распределение рейтингов, жанров, стран и студий.

In [ ]:
categorical_columns = ["rating", "genre", "country", "company", "director", "writer", "star"]

for column in categorical_columns:
    print(f"{column}: уникальных значений = {df[column].nunique()}")
    display(df[column].value_counts().head(10).to_frame("count"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["genre"].value_counts().head(10).sort_values().plot(kind="barh", ax=axes[0], color="#3c73ff")
axes[0].set_title("Топ жанров")
axes[0].set_xlabel("Количество фильмов")

df["country"].value_counts().head(10).sort_values().plot(kind="barh", ax=axes[1], color="#ffb84d")
axes[1].set_title("Топ стран производства")
axes[1].set_xlabel("Количество фильмов")

plt.tight_layout()
plt.show()

## ROI и окупаемость

Для бизнес-интерпретации дополнительно рассчитаем:

- `predicted_profit = gross - budget`;
- `roi = (gross - budget) / budget`;
- `payback_ratio = gross / budget`.

In [ ]:
analysis_df = df.copy()
analysis_df["profit"] = analysis_df["gross"] - analysis_df["budget"]
analysis_df["roi"] = analysis_df["profit"] / analysis_df["budget"]
analysis_df["payback_ratio"] = analysis_df["gross"] / analysis_df["budget"]

display(analysis_df[["profit", "roi", "payback_ratio"]].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

def success_category(roi):
    if roi < 0:
        return "коммерчески неуспешный"
    if roi < 1:
        return "умеренно успешный"
    if roi < 3:
        return "коммерчески успешный"
    return "высокоуспешный / блокбастер"

analysis_df["success_category"] = analysis_df["roi"].apply(success_category)
analysis_df["success_category"].value_counts()

## Российские фильмы в текущем датасете

Проект ориентирован на русскоязычный интерфейс, но текущий датасет в основном описывает зарубежный рынок. Проверим количество фильмов с `country = Russia`.

In [ ]:
russia_mask = df["country"].str.contains("Russia|Russian|Soviet", case=False, na=False)
print("Количество фильмов, связанных с Россией:", int(russia_mask.sum()))
display(df.loc[russia_mask, ["name", "year", "country", "genre", "budget", "gross", "score", "votes"]])

## Выводы

1. `output.csv` — очищенный датасет без пропусков и дубликатов.
2. Целевая переменная проекта — `gross`, то есть кассовая выручка.
3. Бюджет и количество голосов имеют сильную связь с кассой, но `votes` не всегда доступен до выхода фильма.
4. Датасет почти не содержит российских фильмов, поэтому для отдельной модели российского рынка нужен отдельный набор данных.
5. Есть экстремально малые бюджеты, поэтому в веб-сервисе нужна валидация минимального бюджета.

## Дополнение: сравнение исходного и очищенного датасета

Для описания подготовки данных важно показать, что `output.csv` — не случайный файл, а очищенная версия исходного `movies.csv`.

In [ ]:
raw_path = PROJECT_ROOT / "data" / "movies.csv"
raw_df = pd.read_csv(raw_path)

comparison = pd.DataFrame([
    {
        "dataset": "movies.csv",
        "rows": raw_df.shape[0],
        "columns": raw_df.shape[1],
        "missing_total": int(raw_df.isna().sum().sum()),
        "missing_budget": int(raw_df["budget"].isna().sum()),
        "missing_gross": int(raw_df["gross"].isna().sum()),
    },
    {
        "dataset": "output.csv",
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_total": int(df.isna().sum().sum()),
        "missing_budget": int(df["budget"].isna().sum()),
        "missing_gross": int(df["gross"].isna().sum()),
    },
])
comparison

## Дополнение: диапазон применимости модели

Модель обучена на исторических данных. Если пользователь вводит параметры, которые сильно выходят за диапазон обучающей выборки, прогноз может стать некорректным. Поэтому в интерфейсе и API есть ограничение минимального бюджета.

In [ ]:
range_table = pd.DataFrame({
    "feature": ["budget", "gross", "votes", "runtime", "year"],
    "min": [df[c].min() for c in ["budget", "gross", "votes", "runtime", "year"]],
    "p01": [df[c].quantile(0.01) for c in ["budget", "gross", "votes", "runtime", "year"]],
    "median": [df[c].median() for c in ["budget", "gross", "votes", "runtime", "year"]],
    "p99": [df[c].quantile(0.99) for c in ["budget", "gross", "votes", "runtime", "year"]],
    "max": [df[c].max() for c in ["budget", "gross", "votes", "runtime", "year"]],
})
range_table

## Дополнение: текст для магистерской

Очищенный датасет содержит 5421 запись о фильмах за 1980–2020 годы. В датасете отсутствуют пропуски в обязательных признаках и целевой переменной. Основная целевая переменная — кассовая выручка `gross`. Датасет имеет выраженное смещение в сторону фильмов США, что следует учитывать при интерпретации результатов и при дальнейшем развитии сервиса для российского рынка.